# Week 6 — Gymnasium: Famous Examples Only

**From Conway to LangGraph: Agent Systems for Physicists in the LLM Era**  
Università di Bologna · Dipartimento di Fisica

---

## Learning Objectives

By the end of this notebook you will:
1. Install and navigate the **Gymnasium API** — the standard RL environment interface
2. Understand **CartPole-v1** (continuous state, dense reward) and **FrozenLake-v1** (discrete state, sparse reward) at the physics level
3. Train state-of-the-art agents with **Stable-Baselines3** in three lines of code
4. Visualise and interpret **learning curves** using the physicist's lens
5. Analyse **reward landscapes** as potential energy surfaces
6. Test **Lyapunov stability** of the trained CartPole controller

---

## Prerequisites

Make sure Week 5 notebook is complete — we will reference the Q-learning GridWorld implementation.

```bash
pip install gymnasium stable-baselines3[extra] shimmy matplotlib numpy
```

In [ ]:
!pip install gymnasium stable-baselines3[extra] shimmy matplotlib numpy

In [ ]:
# ── Imports and version check ────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import warnings
warnings.filterwarnings('ignore')

import gymnasium as gym
import stable_baselines3 as sb3
from stable_baselines3 import PPO, DQN, A2C
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.callbacks import EvalCallback, BaseCallback
from stable_baselines3.common.monitor import Monitor

print(f'Gymnasium:         {gym.__version__}')
print(f'Stable-Baselines3: {sb3.__version__}')
print(f'NumPy:             {np.__version__}')

---

## Part 1 — The Gymnasium API

Gymnasium defines the canonical **agent–environment loop**:

```
obs, info  = env.reset()           # initialise
obs, r, terminated, truncated, info = env.step(action)  # 5-tuple
```

Every environment exposes two spaces:
- `env.observation_space` — the set of valid observations
- `env.action_space` — the set of valid actions

This interface is identical whether you are controlling a cartpole, a robot arm, or a trading strategy — the algorithm never needs to know what the environment is.

In [ ]:
# ── 1.1  Inspect the CartPole environment ────────────────────────────────────
env_cp = gym.make('CartPole-v1')

print('=== CartPole-v1 ===')
print(f'Observation space: {env_cp.observation_space}')
print(f'  Shape:   {env_cp.observation_space.shape}')
print(f'  Low:     {env_cp.observation_space.low}')
print(f'  High:    {env_cp.observation_space.high}')
print()
print(f'Action space:      {env_cp.action_space}')
print(f'  N actions:       {env_cp.action_space.n}')
print(f'  Meaning:         0 = push LEFT, 1 = push RIGHT')
print()

# Run one random episode and collect the trajectory
obs, info = env_cp.reset(seed=42)
trajectory = [obs.copy()]
rewards = []

for step in range(500):
    action = env_cp.action_space.sample()  # random policy
    obs, reward, terminated, truncated, info = env_cp.step(action)
    trajectory.append(obs.copy())
    rewards.append(reward)
    if terminated or truncated:
        break

env_cp.close()
trajectory = np.array(trajectory)
print(f'Random episode length: {len(rewards)} steps')
print(f'Total reward:          {sum(rewards):.0f}')

In [ ]:
# ── 1.2  Visualise the random episode trajectory ─────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
fig.patch.set_facecolor('#1A1A1A')

labels  = [r'Position $x$ (m)', r'Velocity $\dot{x}$ (m/s)',
           r'Pole angle $\theta$ (rad)', r'Angular velocity $\dot{\theta}$ (rad/s)']
colors  = ['#4A9ECC', '#E8B931', '#5DBB63', '#E05555']
t = np.arange(len(trajectory))

for i, (ax, label, color) in enumerate(zip(axes.flat, labels, colors)):
    ax.set_facecolor('#2D2D2D')
    ax.plot(t, trajectory[:, i], color=color, lw=1.5)
    ax.set_xlabel('Step', color='#888888')
    ax.set_ylabel(label, color=color)
    ax.tick_params(colors='#888888')
    for spine in ax.spines.values():
        spine.set_color('#444444')
    if i == 2:  # pole angle: draw termination lines
        ax.axhline(0.2094, color='red', ls='--', alpha=0.5, label='±12° limit')
        ax.axhline(-0.2094, color='red', ls='--', alpha=0.5)
        ax.legend(facecolor='#1A1A1A', labelcolor='white', framealpha=0.7)

fig.suptitle('CartPole-v1: Random Policy Trajectory', color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('cartpole_random_trajectory.png', dpi=120, bbox_inches='tight', facecolor='#1A1A1A')
plt.show()

In [ ]:
# ── 1.3  Inspect the FrozenLake environment ──────────────────────────────────
env_fl = gym.make('FrozenLake-v1', is_slippery=False)

print('=== FrozenLake-v1 (deterministic) ===')
print(f'Observation space: {env_fl.observation_space}')
print(f'  N states:        {env_fl.observation_space.n}  (16 cells of a 4×4 grid)')
print()
print(f'Action space:      {env_fl.action_space}')
print(f'  N actions:       {env_fl.action_space.n}  (0=L, 1=D, 2=R, 3=U)')
print()
print('Grid map:')
print('  S F F F   (S=Start, F=Frozen, H=Hole, G=Goal)')
print('  F H F H')
print('  F F F H')
print('  H F F G')
print()
print('Reward structure: R=1 at GOAL only — this is a SPARSE reward problem!')

# Run a handcrafted optimal trajectory (deterministic mode)
obs, info = env_fl.reset(seed=0)
# Optimal path: right, right, right, down, down, down, down → goal
# Actually: down, down, right, right, right, down, right
optimal_actions = [1, 2, 1, 1, 2, 2, 1]  # L=0, D=1, R=2, U=3
total_r = 0
for a in optimal_actions:
    obs, r, term, trunc, info = env_fl.step(a)
    total_r += r
    if term:
        break
env_fl.close()
print(f'\nHandcrafted path reward: {total_r}  (reached goal: {total_r > 0})')

---

## Part 2 — Training with Stable-Baselines3

Stable-Baselines3 (SB3) provides clean, well-tested implementations of major RL algorithms. The interface is uniform: every algorithm has `.learn()`, `.predict()`, `.save()`, `.load()`.

**For CartPole** we use **PPO** (Proximal Policy Optimization) — the standard on-policy actor-critic algorithm. It works with continuous state spaces via a neural network policy.

**For FrozenLake** we use **DQN** (Deep Q-Network) — naturally suited for discrete action spaces, and its Q-values directly correspond to the Bellman equation from Week 5.

In [ ]:
# ── 2.1  Callback to record learning curves ──────────────────────────────────
class LearningCurveCallback(BaseCallback):
    """Records episode rewards and policy entropy during training."""
    def __init__(self):
        super().__init__()
        self.episode_rewards = []
        self.episode_lengths = []
        self.timesteps = []

    def _on_step(self) -> bool:
        # SB3 stores episode info in self.locals['infos']
        for info in self.locals.get('infos', []):
            if 'episode' in info:
                self.episode_rewards.append(info['episode']['r'])
                self.episode_lengths.append(info['episode']['l'])
                self.timesteps.append(self.num_timesteps)
        return True

In [ ]:
# ── 2.2  Train PPO on CartPole ────────────────────────────────────────────────
# Note: wrap in Monitor for automatic episode recording
env_train_cp = Monitor(gym.make('CartPole-v1'))

cb_cp = LearningCurveCallback()

model_cp = PPO(
    'MlpPolicy',
    env_train_cp,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    verbose=0,
    seed=42
)

print('Training PPO on CartPole-v1 (100 000 steps)...')
model_cp.learn(total_timesteps=100_000, callback=cb_cp)
model_cp.save('cartpole_ppo')
print('Done. Model saved to cartpole_ppo.zip')

# Quick evaluation
mean_r, std_r = evaluate_policy(model_cp, gym.make('CartPole-v1'), n_eval_episodes=20, deterministic=True)
print(f'\nCartPole evaluation: {mean_r:.1f} ± {std_r:.1f}  (max=500)')

In [ ]:
# ── 2.3  Train DQN on FrozenLake ──────────────────────────────────────────────
env_train_fl = Monitor(gym.make('FrozenLake-v1', is_slippery=False))

cb_fl = LearningCurveCallback()

model_fl = DQN(
    'MlpPolicy',
    env_train_fl,
    learning_rate=1e-3,
    buffer_size=10_000,
    learning_starts=500,
    batch_size=32,
    gamma=0.99,
    exploration_fraction=0.5,
    exploration_final_eps=0.02,
    verbose=0,
    seed=42
)

print('Training DQN on FrozenLake-v1 (50 000 steps)...')
model_fl.learn(total_timesteps=50_000, callback=cb_fl)
model_fl.save('frozenlake_dqn')
print('Done.')

mean_r, std_r = evaluate_policy(model_fl, gym.make('FrozenLake-v1', is_slippery=False),
                                n_eval_episodes=20, deterministic=True)
print(f'\nFrozenLake evaluation: {mean_r:.2f} ± {std_r:.2f}  (1.0 = always reaches goal)')

In [ ]:
# ── 2.4  Plot learning curves side by side ───────────────────────────────────
def smooth(x, w=30):
    """Moving average smoothing."""
    if len(x) < w:
        return np.array(x)
    return np.convolve(np.array(x, dtype=float), np.ones(w)/w, mode='valid')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#1A1A1A')

# CartPole
ax = axes[0]
ax.set_facecolor('#2D2D2D')
r_cp = np.array(cb_cp.episode_rewards)
t_cp = np.array(cb_cp.timesteps)
ax.plot(t_cp, r_cp, color='#4A9ECC', alpha=0.25, lw=0.8, label='Raw')
if len(r_cp) >= 30:
    ax.plot(t_cp[29:], smooth(r_cp), color='#4A9ECC', lw=2, label='Smoothed (30 ep)')
ax.axhline(500, color='#E8B931', ls='--', lw=1, label='Max (solved)')
ax.set_xlabel('Timesteps', color='#888888')
ax.set_ylabel('Episode Reward', color='#4A9ECC')
ax.set_title('CartPole-v1  —  PPO', color='white', fontweight='bold')
ax.legend(facecolor='#1A1A1A', labelcolor='white')
ax.tick_params(colors='#888888')
for sp in ax.spines.values(): sp.set_color('#444444')

# FrozenLake
ax = axes[1]
ax.set_facecolor('#2D2D2D')
r_fl = np.array(cb_fl.episode_rewards)
t_fl = np.array(cb_fl.timesteps)
ax.plot(t_fl, r_fl, color='#5DBB63', alpha=0.25, lw=0.8, label='Raw (0=fail, 1=goal)')
# Compute rolling success rate
if len(r_fl) >= 50:
    success = smooth(r_fl, 50)
    ax.plot(t_fl[49:], success, color='#5DBB63', lw=2, label='Success rate (50 ep)')
ax.axhline(1.0, color='#E8B931', ls='--', lw=1, label='Perfect')
ax.set_xlabel('Timesteps', color='#888888')
ax.set_ylabel('Episode Reward / Success Rate', color='#5DBB63')
ax.set_title('FrozenLake-v1  —  DQN', color='white', fontweight='bold')
ax.legend(facecolor='#1A1A1A', labelcolor='white')
ax.tick_params(colors='#888888')
for sp in ax.spines.values(): sp.set_color('#444444')

fig.suptitle('Learning Curves: CartPole (dense reward) vs FrozenLake (sparse reward)',
             color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('learning_curves.png', dpi=120, bbox_inches='tight', facecolor='#1A1A1A')
plt.show()

print(f'CartPole: {len(r_cp)} episodes trained')
print(f'FrozenLake: {len(r_fl)} episodes trained')

---

## Part 3 — Reward Landscapes as Potential Energy Surfaces

In classical mechanics the action principle selects trajectories that extremise the action $S = \int L \, dt$. In RL the agent selects actions to maximise the **return** $G_t = \sum_k \gamma^k r_{t+k}$.

We can interpret the return as **negative potential energy**: the agent is a particle rolling toward the energy minimum in (policy-space) state-action landscape. This section makes that analogy concrete.

In [ ]:
# ── 3.1  CartPole reward landscape vs pole angle ──────────────────────────────
#
# The instant reward is +1 for every step NOT terminated.
# For a single step: r(θ) = 1  if |θ| < 0.2094 rad,  else 0
# But the *value function* V(s) ~ E[G_t | s] tells us how much
# future reward to expect from state s — that IS the potential energy.
#
# Here we estimate V(θ) empirically by rolling out the trained policy
# from many initial pole angles.

def estimate_value(model, theta_init, n_episodes=20):
    """Estimate V(θ₀) = expected return when starting at pole angle θ₀."""
    env = gym.make('CartPole-v1')
    returns = []
    for _ in range(n_episodes):
        obs, _ = env.reset()
        obs[2] = theta_init  # override pole angle
        G, gamma = 0.0, 0.99
        k = 0
        while True:
            action, _ = model.predict(obs[np.newaxis], deterministic=True)
            obs, r, term, trunc, _ = env.step(int(action))
            G += gamma**k * r
            k += 1
            if term or trunc or k > 500:
                break
        returns.append(G)
    env.close()
    return np.mean(returns)

theta_vals = np.linspace(-0.18, 0.18, 19)  # within termination limit
V_theta = [estimate_value(model_cp, th, n_episodes=10) for th in theta_vals]

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#1A1A1A')
ax.set_facecolor('#2D2D2D')

ax.plot(np.degrees(theta_vals), V_theta, 'o-', color='#4A9ECC', lw=2, ms=6)
ax.fill_between(np.degrees(theta_vals), V_theta, alpha=0.15, color='#4A9ECC')
ax.axvline(0, color='#E8B931', ls=':', lw=1.5, label='Equilibrium θ=0')
ax.axvline(12, color='#E05555', ls='--', lw=1.5, label='Termination ±12°')
ax.axvline(-12, color='#E05555', ls='--', lw=1.5)
ax.set_xlabel(r'Initial pole angle $\theta_0$ (degrees)', color='#888888', fontsize=12)
ax.set_ylabel(r'Expected return $V(\theta_0)$  [= −potential energy]', color='#4A9ECC', fontsize=12)
ax.set_title('Reward Landscape: CartPole Value Function vs Initial Pole Angle',
             color='white', fontsize=13, fontweight='bold')
ax.legend(facecolor='#1A1A1A', labelcolor='white')
ax.tick_params(colors='#888888')
for sp in ax.spines.values(): sp.set_color('#444444')

plt.tight_layout()
plt.savefig('reward_landscape.png', dpi=120, bbox_inches='tight', facecolor='#1A1A1A')
plt.show()

print('Physical interpretation:')
print(f'  V(0°)  = {estimate_value(model_cp, 0.0, 30):.1f}  (near-perfect control near equilibrium)')
print(f'  V(10°) = {estimate_value(model_cp, 0.175, 30):.1f}  (near termination boundary)')

In [ ]:
# ── 3.2  FrozenLake value function heatmap ────────────────────────────────────
import torch

# Extract Q-values from the trained DQN network
# DQN in SB3 stores q_net which maps observation → Q-values for all actions
state_obs = torch.eye(16, dtype=torch.float32)  # one-hot encoding (if needed)
# For discrete obs, SB3 DQN uses the raw integer — we need to check the network input
# In SB3, FrozenLake obs is a scalar integer; with MlpPolicy it's fed as float
state_inputs = torch.arange(16, dtype=torch.float32).unsqueeze(1)

with torch.no_grad():
    # Access the q_net via model internals
    features = model_fl.policy.q_net.features_extractor(state_inputs)
    q_vals = model_fl.policy.q_net.q_net(features)

q_np = q_vals.numpy()  # shape (16, 4)
V_star = q_np.max(axis=1).reshape(4, 4)  # V*(s) = max_a Q*(s,a)
best_a = q_np.argmax(axis=1).reshape(4, 4)

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#1A1A1A')

action_arrows = {0: '←', 1: '↓', 2: '→', 3: '↑'}
grid_map = ['SFFF', 'FHFH', 'FFFH', 'HFFG']
special = {'S': '#1E4A2C', 'H': '#4A1A1A', 'G': '#1E3A4C', 'F': None}

# Panel 1: V*(s) heatmap
ax = axes[0]
masked_V = np.ma.masked_where(np.array([[c in 'HG' for c in row] for row in grid_map]), V_star)
im = ax.imshow(V_star, cmap='Blues', aspect='equal')
ax.set_facecolor('#2D2D2D')
plt.colorbar(im, ax=ax, label='V*(s) — expected return')
for r in range(4):
    for c in range(4):
        cell = grid_map[r][c]
        if cell == 'H':
            ax.add_patch(plt.Rectangle((c-0.5, r-0.5), 1, 1, color='#4A1A1A'))
            ax.text(c, r, 'H', ha='center', va='center', color='#E05555', fontsize=14, fontweight='bold')
        elif cell == 'G':
            ax.add_patch(plt.Rectangle((c-0.5, r-0.5), 1, 1, color='#1E3A4C'))
            ax.text(c, r, '★', ha='center', va='center', color='#E8B931', fontsize=16)
        else:
            ax.text(c, r, f'{V_star[r,c]:.2f}', ha='center', va='center',
                   color='white', fontsize=11)
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_title('V*(s) — Bellman Values\n(learned by DQN)', color='white', fontweight='bold')
ax.tick_params(colors='#888888')

# Panel 2: Greedy policy arrows
ax = axes[1]
ax.set_facecolor('#2D2D2D')
ax.imshow(np.ones((4,4)), cmap='Greys', aspect='equal', alpha=0.1)
for r in range(4):
    for c in range(4):
        cell = grid_map[r][c]
        if cell == 'H':
            ax.add_patch(plt.Rectangle((c-0.5, r-0.5), 1, 1, color='#4A1A1A'))
            ax.text(c, r, 'H', ha='center', va='center', color='#E05555', fontsize=14, fontweight='bold')
        elif cell == 'G':
            ax.add_patch(plt.Rectangle((c-0.5, r-0.5), 1, 1, color='#1E3A4C'))
            ax.text(c, r, 'G', ha='center', va='center', color='#E8B931', fontsize=16, fontweight='bold')
        elif cell == 'S':
            ax.add_patch(plt.Rectangle((c-0.5, r-0.5), 1, 1, color='#1E4A2C'))
            ax.text(c, r, action_arrows[best_a[r,c]], ha='center', va='center',
                   color='#5DBB63', fontsize=20, fontweight='bold')
        else:
            ax.text(c, r, action_arrows[best_a[r,c]], ha='center', va='center',
                   color='#4A9ECC', fontsize=20, fontweight='bold')
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_title('Greedy Policy π*(s)\n(best action per cell)', color='white', fontweight='bold')
ax.tick_params(colors='#888888')

fig.suptitle('FrozenLake-v1: DQN learns the Bellman solution from Week 5',
             color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('frozenlake_values_policy.png', dpi=120, bbox_inches='tight', facecolor='#1A1A1A')
plt.show()

---

## Part 4 — Analysing the Trained CartPole Policy

The PPO policy is a neural network $\pi_\theta(a|s)$ — a black box. We can make it interpretable by probing it as a function of state variables. The key question: **does the learned controller rediscover the optimal Linear-Quadratic Regulator (LQR)?**

For an inverted pendulum, the classical LQR solution has the form:

$$u = -K \begin{pmatrix} x \\ \dot{x} \\ \theta \\ \dot{\theta} \end{pmatrix}$$

where $K$ is the optimal gain matrix. The decision boundary in $(\theta, \dot{\theta})$ space is approximately linear — a straight line through the origin.

In [ ]:
# ── 4.1  Phase portrait of the trained PPO policy ────────────────────────────
theta_range     = np.linspace(-0.18, 0.18, 60)
theta_dot_range = np.linspace(-2.0, 2.0, 60)
TH, TD = np.meshgrid(theta_range, theta_dot_range)

# Query the policy at each (θ, θ̇) point with x=0, ẋ=0
actions = np.zeros_like(TH)
probs_right = np.zeros_like(TH)  # probability of action=RIGHT

for i, th in enumerate(theta_range):
    for j, td in enumerate(theta_dot_range):
        obs = np.array([[0.0, 0.0, th, td]], dtype=np.float32)
        act, _ = model_cp.predict(obs, deterministic=True)
        actions[j, i] = int(act)
        # Get soft probabilities from policy
        with torch.no_grad():
            obs_t = torch.tensor(obs)
            dist = model_cp.policy.get_distribution(obs_t)
            p = dist.distribution.probs.numpy()[0]
            probs_right[j, i] = p[1]  # P(push right)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#1A1A1A')

# Panel 1: Binary decision map
ax = axes[0]
ax.set_facecolor('#2D2D2D')
cmap_br = ListedColormap(['#4A9ECC', '#E05555'])  # blue=LEFT, red=RIGHT
ax.contourf(np.degrees(TH), TD, actions, levels=[-0.5, 0.5, 1.5], cmap=cmap_br, alpha=0.85)
ax.contour(np.degrees(TH), TD, actions, levels=[0.5], colors=['#E8B931'], linewidths=2)
ax.axhline(0, color='white', ls=':', alpha=0.3)
ax.axvline(0, color='white', ls=':', alpha=0.3)
left_patch = mpatches.Patch(color='#4A9ECC', label='Push LEFT (0)')
right_patch = mpatches.Patch(color='#E05555', label='Push RIGHT (1)')
ax.legend(handles=[left_patch, right_patch], facecolor='#1A1A1A', labelcolor='white')
ax.set_xlabel(r'Pole angle $\theta$ (deg)', color='#888888', fontsize=12)
ax.set_ylabel(r'Angular velocity $\dot{\theta}$ (rad/s)', color='#888888', fontsize=12)
ax.set_title('PPO Policy Decision Boundary\n(x=0, ẋ=0)', color='white', fontweight='bold')
ax.tick_params(colors='#888888')
for sp in ax.spines.values(): sp.set_color('#444444')

# Panel 2: Soft probability surface
ax = axes[1]
ax.set_facecolor('#2D2D2D')
c = ax.contourf(np.degrees(TH), TD, probs_right, levels=20, cmap='RdBu_r', vmin=0, vmax=1)
plt.colorbar(c, ax=ax, label='P(push RIGHT)')
ax.contour(np.degrees(TH), TD, probs_right, levels=[0.5], colors=['#E8B931'], linewidths=2,
           linestyles='--')
ax.axhline(0, color='white', ls=':', alpha=0.3)
ax.axvline(0, color='white', ls=':', alpha=0.3)
ax.set_xlabel(r'Pole angle $\theta$ (deg)', color='#888888', fontsize=12)
ax.set_ylabel(r'Angular velocity $\dot{\theta}$ (rad/s)', color='#888888', fontsize=12)
ax.set_title('PPO Soft Policy P(Right | θ, θ̇)\nDecision boundary ≈ LQR optimal', color='white', fontweight='bold')
ax.tick_params(colors='#888888')
for sp in ax.spines.values(): sp.set_color('#444444')

fig.suptitle('CartPole PPO Phase Portrait: PPO ≈ Rediscovers LQR Controller',
             color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('cartpole_phase_portrait.png', dpi=120, bbox_inches='tight', facecolor='#1A1A1A')
plt.show()

print('The yellow dashed line (decision boundary) should be approximately linear — confirming LQR analogy.')

---

## Part 5 — Lyapunov Stability Analysis

A controller is **Lyapunov stable** around the equilibrium $(\theta^*, \dot{\theta}^*) = (0, 0)$ if there exists a positive definite function $V(s) > 0$ such that

$$\frac{dV}{dt} = V(s_{t+1}) - V(s_t) < 0 \quad \text{whenever} \quad s \neq s^*$$

This is the discrete-time Lyapunov condition. We use the candidate $V(s) = \theta^2 + c \dot{\theta}^2$, which is just the mechanical energy of the linearised pendulum (up to a constant).

In [ ]:
# ── 5.1  Lyapunov stability test on trained PPO ───────────────────────────────
c = 0.01  # weight on angular velocity term; tune to physical scale

def lyapunov_V(obs, c=0.01):
    """V(s) = θ² + c·θ̇²  —  candidate Lyapunov function."""
    theta, theta_dot = obs[2], obs[3]
    return theta**2 + c * theta_dot**2

# Collect many transitions from the trained policy
env_lyap = gym.make('CartPole-v1')
all_dV = []
all_theta = []
all_episodes_V = []

for ep in range(100):
    obs, _ = env_lyap.reset()
    V_history = [lyapunov_V(obs, c)]
    ep_data = []
    for _ in range(500):
        V_before = lyapunov_V(obs, c)
        action, _ = model_cp.predict(obs[np.newaxis], deterministic=True)
        obs_next, r, term, trunc, _ = env_lyap.step(int(action))
        V_after = lyapunov_V(obs_next, c)
        dV = V_after - V_before
        ep_data.append((obs[2], dV))  # (θ, dV)
        V_history.append(V_after)
        obs = obs_next
        if term or trunc:
            break
    all_dV.extend([d[1] for d in ep_data])
    all_theta.extend([d[0] for d in ep_data])
    all_episodes_V.append(V_history)

env_lyap.close()

all_dV = np.array(all_dV)
all_theta = np.array(all_theta)
print(f'Collected {len(all_dV)} transitions from 100 episodes')
print(f'Mean dV/dt = {all_dV.mean():.5f}  (negative = stable on average)')
print(f'Fraction with dV < 0: {(all_dV < 0).mean():.3f}')

In [ ]:
# ── 5.2  Visualise the Lyapunov analysis ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor('#1A1A1A')

# Panel 1: dV vs θ scatter
ax = axes[0]
ax.set_facecolor('#2D2D2D')
# Bin and mean dV by theta
bins = np.linspace(-0.2, 0.2, 25)
bin_idx = np.digitize(all_theta, bins)
bin_centers = (bins[:-1] + bins[1:]) / 2
bin_mean_dV = [all_dV[bin_idx == i+1].mean() if (bin_idx == i+1).sum() > 5 else np.nan
               for i in range(len(bin_centers))]

ax.scatter(np.degrees(all_theta[::3]), all_dV[::3], alpha=0.05, s=2, color='#4A9ECC')
ax.plot(np.degrees(bin_centers), bin_mean_dV, color='#E8B931', lw=2.5, label='Mean dV/dt per bin')
ax.axhline(0, color='white', ls='--', lw=1, alpha=0.7)
ax.axvspan(-7, 7, alpha=0.08, color='#5DBB63', label='Stable region |θ|<7°')
ax.set_xlabel(r'$\theta$ (deg)', color='#888888')
ax.set_ylabel(r'$dV/dt = V(s_{t+1}) - V(s_t)$', color='#4A9ECC')
ax.set_title('Lyapunov Derivative vs Angle', color='white', fontweight='bold')
ax.legend(facecolor='#1A1A1A', labelcolor='white', fontsize=9)
ax.tick_params(colors='#888888')
for sp in ax.spines.values(): sp.set_color('#444444')

# Panel 2: V(t) trajectories for a few episodes
ax = axes[1]
ax.set_facecolor('#2D2D2D')
for i in range(10):
    t = np.arange(len(all_episodes_V[i]))
    ax.plot(t, all_episodes_V[i], lw=1, alpha=0.5, color='#4A9ECC')
ax.set_xlabel('Step', color='#888888')
ax.set_ylabel(r'$V(s) = \theta^2 + c \dot{\theta}^2$', color='#4A9ECC')
ax.set_title('V(t) Along Trajectories\n(should trend downward)', color='white', fontweight='bold')
ax.tick_params(colors='#888888')
for sp in ax.spines.values(): sp.set_color('#444444')

# Panel 3: Histogram of dV values
ax = axes[2]
ax.set_facecolor('#2D2D2D')
ax.hist(all_dV, bins=60, color='#4A9ECC', alpha=0.7, edgecolor='none')
ax.axvline(0, color='#E8B931', lw=2, label=f'Zero boundary')
ax.axvline(all_dV.mean(), color='#5DBB63', lw=2, ls='--', label=f'Mean={all_dV.mean():.5f}')
ax.set_xlabel('dV/dt', color='#888888')
ax.set_ylabel('Count', color='#888888')
ax.set_title(f'Distribution of dV/dt\n{100*(all_dV<0).mean():.1f}% < 0 (stable steps)', color='white', fontweight='bold')
ax.legend(facecolor='#1A1A1A', labelcolor='white')
ax.tick_params(colors='#888888')
for sp in ax.spines.values(): sp.set_color('#444444')

fig.suptitle('Lyapunov Stability Analysis of Trained PPO CartPole Controller',
             color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('lyapunov_analysis.png', dpi=120, bbox_inches='tight', facecolor='#1A1A1A')
plt.show()

---

## Part 6 — Algorithm Comparison

Different RL algorithms make different tradeoffs. Here we train both **PPO** and **A2C** on CartPole and compare learning speed, stability, and final performance.

In [ ]:
# ── 6.1  PPO vs A2C on CartPole ───────────────────────────────────────────────
cb_a2c = LearningCurveCallback()
env_a2c = Monitor(gym.make('CartPole-v1'))

model_a2c = A2C('MlpPolicy', env_a2c, learning_rate=7e-4, verbose=0, seed=42)

print('Training A2C on CartPole-v1 (100 000 steps)...')
model_a2c.learn(total_timesteps=100_000, callback=cb_a2c)
print('Done.')

mean_ppo, _ = evaluate_policy(model_cp, gym.make('CartPole-v1'), n_eval_episodes=20, deterministic=True)
mean_a2c, _ = evaluate_policy(model_a2c, gym.make('CartPole-v1'), n_eval_episodes=20, deterministic=True)
print(f'\nPPO: {mean_ppo:.1f}/500')
print(f'A2C: {mean_a2c:.1f}/500')

In [ ]:
# ── 6.2  Compare learning curves ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor('#1A1A1A')
ax.set_facecolor('#2D2D2D')

for (rewards, timesteps, color, name) in [
    (cb_cp.episode_rewards,  cb_cp.timesteps,  '#4A9ECC', 'PPO'),
    (cb_a2c.episode_rewards, cb_a2c.timesteps, '#E8B931', 'A2C'),
]:
    r_arr = np.array(rewards)
    t_arr = np.array(timesteps)
    ax.plot(t_arr, r_arr, alpha=0.15, lw=0.6, color=color)
    if len(r_arr) >= 30:
        ax.plot(t_arr[29:], smooth(r_arr, 30), lw=2.5, color=color, label=name)

ax.axhline(500, color='#5DBB63', ls='--', lw=1.5, label='Solved (500)')
ax.set_xlabel('Timesteps', color='#888888', fontsize=12)
ax.set_ylabel('Episode Reward (smoothed)', color='white', fontsize=12)
ax.set_title('CartPole: PPO vs A2C Learning Curve Comparison', color='white', fontsize=13, fontweight='bold')
ax.legend(facecolor='#1A1A1A', labelcolor='white', fontsize=11)
ax.tick_params(colors='#888888')
for sp in ax.spines.values(): sp.set_color('#444444')

plt.tight_layout()
plt.savefig('ppo_vs_a2c.png', dpi=120, bbox_inches='tight', facecolor='#1A1A1A')
plt.show()

print('Observations:')
print('  PPO  — stable learning, smooth curves due to trust-region clipping')
print('  A2C  — faster initial learning, noisier (no replay buffer, on-policy)')

---

## Part 7 — FrozenLake: Sparse vs Dense Reward

Sparse rewards are the core challenge in RL. Here we train on the stochastic version of FrozenLake and compare convergence behaviour.

In [ ]:
# ── 7.1  FrozenLake deterministic vs stochastic ───────────────────────────────
cb_slip = LearningCurveCallback()
env_slip = Monitor(gym.make('FrozenLake-v1', is_slippery=True))

model_slip = DQN(
    'MlpPolicy', env_slip,
    learning_rate=5e-4,
    buffer_size=50_000,
    learning_starts=1000,
    batch_size=64,
    gamma=0.99,
    exploration_fraction=0.8,
    exploration_final_eps=0.05,
    verbose=0, seed=42
)

print('Training DQN on FrozenLake-v1 (stochastic, 200 000 steps)...')
model_slip.learn(total_timesteps=200_000, callback=cb_slip)
print('Done.')

mean_det, _ = evaluate_policy(model_fl,   gym.make('FrozenLake-v1', is_slippery=False),
                              n_eval_episodes=100, deterministic=True)
mean_sto, _ = evaluate_policy(model_slip, gym.make('FrozenLake-v1', is_slippery=True),
                              n_eval_episodes=100, deterministic=True)
print(f'\nDeterministic FrozenLake success rate: {mean_det*100:.0f}%')
print(f'Stochastic FrozenLake success rate:    {mean_sto*100:.0f}%')

In [ ]:
# ── 7.2  Convergence comparison ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor('#1A1A1A')
ax.set_facecolor('#2D2D2D')

for (rewards, timesteps, color, name, w) in [
    (cb_fl.episode_rewards,   cb_fl.timesteps,   '#5DBB63', 'Deterministic', 100),
    (cb_slip.episode_rewards, cb_slip.timesteps, '#E05555', 'Stochastic',    200),
]:
    r_arr = np.array(rewards, dtype=float)
    t_arr = np.array(timesteps)
    ax.plot(t_arr, r_arr, alpha=0.08, lw=0.5, color=color)
    if len(r_arr) >= w:
        ax.plot(t_arr[w-1:], smooth(r_arr, w), lw=2.5, color=color, label=name)

ax.axhline(1.0, color='#E8B931', ls='--', lw=1.5, label='Perfect (R=1)')
ax.set_xlabel('Timesteps', color='#888888', fontsize=12)
ax.set_ylabel('Success Rate', color='white', fontsize=12)
ax.set_title('FrozenLake: Deterministic vs Stochastic (slippery)\nSparse reward makes stochastic case dramatically harder',
             color='white', fontsize=12, fontweight='bold')
ax.legend(facecolor='#1A1A1A', labelcolor='white', fontsize=11)
ax.tick_params(colors='#888888')
for sp in ax.spines.values(): sp.set_color('#444444')

plt.tight_layout()
plt.savefig('frozenlake_det_vs_stoch.png', dpi=120, bbox_inches='tight', facecolor='#1A1A1A')
plt.show()

print('Physics interpretation:')
print('  Deterministic: the agent has a well-defined policy path to follow')
print('  Stochastic:    each action is a biased random walk — like Brownian motion')
print('  The success probability scales as the probability of reaching G before H')
print('  in a 2D lattice random walk — can be computed from gambler\'s ruin theorem!')

---

## Summary

| Concept | CartPole-v1 | FrozenLake-v1 |
|---------|-------------|---------------|
| State space | ℝ⁴ (continuous) | 𝕫₁₆ (discrete) |
| Reward | Dense (+1/step) | Sparse (+1 goal only) |
| Transitions | Deterministic ODE | Stochastic (slippery) |
| Best algorithm | PPO / A2C | DQN |
| Physics angle | Inverted pendulum / Lyapunov | Random walk / gambler's ruin |
| Week 5 link | Actor-critic ≈ Policy gradient | DQN Q-values = Bellman solution |

### Key Findings

1. **Reward landscape** is a potential energy surface — dense rewards are convex and easy to navigate; sparse rewards are flat with a delta-peak and nearly zero gradient.
2. **PPO rediscovers LQR** — the neural network decision boundary in (θ, θ̇) space is approximately linear, confirming the classical optimal control result.
3. **Lyapunov stability** can be tested empirically — the trained controller satisfies dV/dt < 0 on average in the stable region.
4. **Stochastic transitions** dramatically increase sample complexity — the success rate gap between deterministic and stochastic FrozenLake quantifies the cost of environment noise.

---

## Exercises

### Exercise 1 — Run + Solve CartPole  *(30 min)*

Re-run the PPO training above with different `total_timesteps` values: 10 000, 25 000, 50 000, 100 000. For each, record the final mean reward from `evaluate_policy`. Plot final performance vs training budget. At what timestep does the agent first reliably achieve >450/500?

### Exercise 2 — FrozenLake Sparse Reward Challenge  *(40 min)*

Compare DQN performance on FrozenLake with `is_slippery=False` vs `is_slippery=True`. Already done in Part 7, but now:
- Apply the **gambler's ruin theorem** to estimate the probability that a random walk from S reaches G before any H on the 4×4 grid.
- Compare this baseline probability to your DQN success rate on the stochastic case.
- What is the improvement factor achieved by learning?

### Exercise 3 — Reward Landscape Visualisation  *(45 min)*

Extend the reward landscape analysis (Part 3.1) to **two dimensions**: vary both θ (pole angle) and x (cart position). Create a 2D heatmap of V(x, θ) with ẋ=0, θ̇=0. Is the landscape separable in x and θ? Does V(x, θ) ≈ V(θ) (i.e., does the position matter much)?

### Exercise 4 ★ — Lyapunov Stability Boundary  *(60 min)*

Using the Lyapunov analysis from Part 5:
1. For each pole angle bin, compute the mean dV/dt.
2. Find the **stability boundary** θ* where mean dV/dt changes sign from negative to positive.
3. Compare θ* to the termination limit of ±12°. Is the stable region smaller or larger than the termination region?
4. Explain physically why dV/dt > 0 near the termination boundary.